In [67]:
import requests
import os
import sqlite3
import pandas as pd
import getdata



conn = sqlite3.connect(r"C:\Users\james\OneDrive\Desktop\CIT460Project\league.db")
API_KEY = os.getenv("RIOT_API")  
REGION = "americas"
HEADERS = {"X-Riot-Token": API_KEY}

def safe_request(url, params=None):
    
    while True:
        r = requests.get(url, headers=HEADERS, params=params)
        if r.status_code == 429:  
            retry_after = int(r.headers.get("Retry-After", 5))
            print(f"Rate limited. Sleeping for {retry_after} seconds.")
            import time; time.sleep(retry_after)
            continue
        if r.status_code != 200:
            print(f"Error {r.status_code} for URL: {url}")
            return None
        return r.json()

In [68]:
query = "SELECT * FROM participants"
df = pd.read_sql_query(query, conn)
pd.set_option('display.max_columns', None)


pd.set_option('display.max_rows', 100)

conn.close()

In [69]:
print(df.head(10))

   id        match_id                                              puuid  \
0   1  NA1_5501660233  xEV-uEvMfQWj7RhuT1NV6pcPpk8kWcOdGrH25wZTdRvw0i...   
1   2  NA1_5501660233  QZANPB_Kp2yr3debaW6cdGty4DqhPD71pqnfbc9Ewmd8VR...   
2   3  NA1_5501660233  qEaOpZMEHHgtKW6PBo0KGC5Zzwmo2XFd7QBKCwksbzmIYL...   
3   4  NA1_5501660233  IgdT6p_UHIpiRR0NtcbccDCdKENk2D6K3QqNVJS2XIsCa0...   
4   5  NA1_5501660233  jPezXkP2Chi4Z7UCN8apT5WYs9wwaDzUgzlHGUKi-QDRfD...   
5   6  NA1_5501660233  LmDx4h4TdkgM3hrRenx6wP133pLOhcNSGJnS5cluq4rF-a...   
6   7  NA1_5501660233  gfIGmkcEE4a8mACteQT-GP8hIkba84t9DagPgKv7-TQZMg...   
7   8  NA1_5501660233  -VwfGhk6pWIsSWm1HeFeMpbMVv-LpCgBjeAZJqPVz02T8w...   
8   9  NA1_5501660233  xl3TfG1Xz0SuOyQpNKcksaOkh2c4cD68YWsuxsJGCpXv1u...   
9  10  NA1_5501660233  DX4_1tiKOda9B3rn3rLnIt_sDf5UNmwSGMJyMt0vB9ywi8...   

    champion position  win  kills  deaths  assists  gold_earned   cs  
0  TahmKench      TOP    1      4      11       16        15573  196  
1     Rammus   JUNGLE

In [70]:
def get_match(match_id):
    url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    return safe_request(url)
def get_team_objectives(match_data):
    result = {}
    for team in match_data["info"]["teams"]:
        team_id = team["teamId"]
        result[team_id] = {
            "win": team["win"],
            "first_blood": team["objectives"]["champion"]["first"],
            "first_tower": team["objectives"]["tower"]["first"],
            "first_inhibitor": team["objectives"]["inhibitor"]["first"],
            "first_baron": team["objectives"]["baron"]["first"],
            "first_dragon": team["objectives"]["dragon"]["first"],
            "tower_kills": team["objectives"]["tower"]["kills"],
            "inhibitor_kills": team["objectives"]["inhibitor"]["kills"],
            "baron_kills": team["objectives"]["baron"]["kills"],
            "dragon_kills": team["objectives"]["dragon"]["kills"],
            "rift_herald_kills": team["objectives"]["riftHerald"]["kills"],
        }
    return result

def get_participant_perks(match_data):
    result = {}
    for p in match_data["info"]["participants"]:
        perks = p.get("perks", {})
        styles = perks.get("styles", [])
        main = styles[0] if len(styles) > 0 else {}
        secondary = styles[1] if len(styles) > 1 else {}

        result[p["puuid"]] = {
            "main_style": main.get("style"),
            "main_keystone": main.get("selections", [{}])[0].get("perk"),
            "main_runes": [s.get("perk") for s in main.get("selections", [{}])[1:5]],
            "secondary_style": secondary.get("style"),
            "secondary_runes": [s.get("perk") for s in secondary.get("selections", [{}])[:2]],
            "stat_perks": perks.get("statPerks", [None, None, None])
        }
    return result

In [71]:
get_match('NA1_5501660233')

{'metadata': {'dataVersion': '2',
  'matchId': 'NA1_5501660233',
  'participants': ['xEV-uEvMfQWj7RhuT1NV6pcPpk8kWcOdGrH25wZTdRvw0iQc8wIVPUkhwgznFpq3MHAnrjba_qLL7Q',
   'QZANPB_Kp2yr3debaW6cdGty4DqhPD71pqnfbc9Ewmd8VRIb9lDUPkn5PyhT9H4B5huX_4rkrnvDtA',
   'qEaOpZMEHHgtKW6PBo0KGC5Zzwmo2XFd7QBKCwksbzmIYL_DR_VIaXMjz9RqlA9FD6NyM30Uonc2pg',
   'IgdT6p_UHIpiRR0NtcbccDCdKENk2D6K3QqNVJS2XIsCa0Xb80jRmJCAqP7aQCbTd8ux2NVcV9fJ8w',
   'jPezXkP2Chi4Z7UCN8apT5WYs9wwaDzUgzlHGUKi-QDRfD0nkq_Twcu4ygqBS9xpst3hyWav_weYSw',
   'LmDx4h4TdkgM3hrRenx6wP133pLOhcNSGJnS5cluq4rF-abUJ3T2QwzAsU2kbZwn34mhy1GwKOxGwA',
   'gfIGmkcEE4a8mACteQT-GP8hIkba84t9DagPgKv7-TQZMg79DHtgEKZw_yZ4stqcwn4MlZQ5I_WsOA',
   '-VwfGhk6pWIsSWm1HeFeMpbMVv-LpCgBjeAZJqPVz02T8wf39-FQEYXuRO0jubDDq7d2R3c_Db0OvQ',
   'xl3TfG1Xz0SuOyQpNKcksaOkh2c4cD68YWsuxsJGCpXv1u7yzI-qnKT_JrYHIML_7EaCAN2_BLyB8g',
   'DX4_1tiKOda9B3rn3rLnIt_sDf5UNmwSGMJyMt0vB9ywi8zHl-sfHlUAfDwDwjUNQonWfdM8CzXXXw']},
 'info': {'endOfGameResult': 'GameComplete',
  'gameCreation': 1772

In [72]:
match_data = getdata.get_match_data("NA1_5501660233", "RGAPI-0caa77ff-24ab-4766-8228-c7de83862976")

match_data["info"]["teams"]

[{'bans': [{'championId': 53, 'pickTurn': 1},
   {'championId': 51, 'pickTurn': 2},
   {'championId': 41, 'pickTurn': 3},
   {'championId': 90, 'pickTurn': 4},
   {'championId': 106, 'pickTurn': 5}],
  'feats': {'EPIC_MONSTER_KILL': {'featState': 0},
   'FIRST_BLOOD': {'featState': 0},
   'FIRST_TURRET': {'featState': 0}},
  'objectives': {'atakhan': {'first': False, 'kills': 0},
   'baron': {'first': True, 'kills': 1},
   'champion': {'first': False, 'kills': 48},
   'dragon': {'first': True, 'kills': 5},
   'horde': {'first': False, 'kills': 0},
   'inhibitor': {'first': False, 'kills': 1},
   'riftHerald': {'first': False, 'kills': 0},
   'tower': {'first': False, 'kills': 8}},
  'teamId': 100,
  'win': True},
 {'bans': [{'championId': 51, 'pickTurn': 6},
   {'championId': 233, 'pickTurn': 7},
   {'championId': 267, 'pickTurn': 8},
   {'championId': 157, 'pickTurn': 9},
   {'championId': 122, 'pickTurn': 10}],
  'feats': {'EPIC_MONSTER_KILL': {'featState': 0},
   'FIRST_BLOOD': {'fe

In [73]:
get_participant_perks(match_data)
# ue riot dragon https://ddragon.leagueoflegends.com/cdn/<patch>/data/en_US/runesReforged.json to translate this

{'xEV-uEvMfQWj7RhuT1NV6pcPpk8kWcOdGrH25wZTdRvw0iQc8wIVPUkhwgznFpq3MHAnrjba_qLL7Q': {'main_style': 8400,
  'main_keystone': 8437,
  'main_runes': [8401, 8444, 8453],
  'secondary_style': 8000,
  'secondary_runes': [9111, 8017],
  'stat_perks': {'defense': 5011, 'flex': 5008, 'offense': 5005}},
 'QZANPB_Kp2yr3debaW6cdGty4DqhPD71pqnfbc9Ewmd8VRIb9lDUPkn5PyhT9H4B5huX_4rkrnvDtA': {'main_style': 8400,
  'main_keystone': 8439,
  'main_runes': [8463, 8429, 8242],
  'secondary_style': 8000,
  'secondary_runes': [9104, 9111],
  'stat_perks': {'defense': 5001, 'flex': 5001, 'offense': 5005}},
 'qEaOpZMEHHgtKW6PBo0KGC5Zzwmo2XFd7QBKCwksbzmIYL_DR_VIaXMjz9RqlA9FD6NyM30Uonc2pg': {'main_style': 8000,
  'main_keystone': 8010,
  'main_runes': [8009, 9105, 8299],
  'secondary_style': 8200,
  'secondary_runes': [8275, 8210],
  'stat_perks': {'defense': 5011, 'flex': 5008, 'offense': 5008}},
 'IgdT6p_UHIpiRR0NtcbccDCdKENk2D6K3QqNVJS2XIsCa0Xb80jRmJCAqP7aQCbTd8ux2NVcV9fJ8w': {'main_style': 8200,
  'main_keysto

In [74]:
get_team_objectives(match_data)

{100: {'win': True,
  'first_blood': False,
  'first_tower': False,
  'first_inhibitor': False,
  'first_baron': True,
  'first_dragon': True,
  'tower_kills': 8,
  'inhibitor_kills': 1,
  'baron_kills': 1,
  'dragon_kills': 5,
  'rift_herald_kills': 0},
 200: {'win': False,
  'first_blood': True,
  'first_tower': True,
  'first_inhibitor': True,
  'first_baron': False,
  'first_dragon': False,
  'tower_kills': 9,
  'inhibitor_kills': 2,
  'baron_kills': 1,
  'dragon_kills': 2,
  'rift_herald_kills': 1}}